In [81]:
import yaml
import copy
from datetime import datetime,timezone,timedelta

In [82]:
def load_yaml(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)
    
weight_config  = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml") 

### Input

In [83]:
op1 = {
    'section': 'Profile',
    'scores': {
        'ContentQuality': {
            'score': 1,
            'feedback': "xxx"},
        'Completeness': {
            'score': 1,
            'feedback': "yyy"}
        },
    'session_feedback': 'zzz'
}
op3 = {
    'section': 'Education',
    'scores': {
        'RoleRelevance': {
            'score': 5,
            'feedback': 'abc'},
        'Completeness': {
        'score': 5,
        'feedback': "def"}
        },
    'session_feedback': 'ghi'
}

### Aggregator

In [84]:
def aggregate(llm_output:dict):
    '''
    Convert, reshape, transform output format that we got from llm
    '''
    llm_output   = llm_output               # Op
    section_name = llm_output["section"]    # Get section name such as Profile, Summary, ..., Skills
    section_config_scores   = weight_config["weights"][section_name]   # Get max score in weight.yaml config
    scaled_criteria_scores  = {}            # Output dictionary
    total_section_raw_score = 0.0           # Accumulate raw score from every criteria
    total_section_max_score = 0.0           # Accumulate maximum score from every criteria that posible
    scores_copy  = copy.deepcopy(llm_output["scores"])         # Protect multiple mutation when we run more than one time

    for criteria,body in scores_copy.items():  # Loop with op...
        raw_llm_score = body["score"]       # raw score for every criteria each section
        if raw_llm_score == 0:              # If LLM detect empty value then max_score should be 0 (don't calcualte it)
            max_score_from_config = 0       # score = 0 instead maximum score
        else:
            max_score_from_config = section_config_scores[criteria] # Max score in weight.yaml (default=10)
        scaled_score = (raw_llm_score / 5) * max_score_from_config  # raw_score/max scale score(5) x max score in weight.yaml(10)
        body["score"] = scaled_score            # Replace new scaled score in body
        scaled_criteria_scores[criteria] = body # Create new dict (Op -> S)
        total_section_raw_score = total_section_raw_score + scaled_score # Accumulate scaled raw score from each criteria in each seciton
        total_section_max_score = total_section_max_score + max_score_from_config # Accumulate full score from config file that Llm not detect 0
    return {
            "section": section_name,
            "total_section_raw_score":total_section_raw_score,
            "total_section_max_score":total_section_max_score,
            "scores":scaled_criteria_scores,
            "session_feedback":llm_output['session_feedback']
        }


In [85]:
s1 = aggregate(op1)
s2 = aggregate(op3)

In [86]:
s1

{'section': 'Profile',
 'total_section_raw_score': 4.0,
 'total_section_max_score': 20.0,
 'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
  'Completeness': {'score': 2.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [87]:
s2

{'section': 'Education',
 'total_section_raw_score': 20.0,
 'total_section_max_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
  'Completeness': {'score': 10.0, 'feedback': 'def'}},
 'session_feedback': 'ghi'}

### GlobalAggregator

In [88]:
section_outputs = [s1,s2]
timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
model_config    = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
weight_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
prompt_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml") 
config_lang     = prompt_config['Language_output_style']["en"]

In [89]:
section_outputs

[{'section': 'Profile',
  'total_section_raw_score': 4.0,
  'total_section_max_score': 20.0,
  'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
   'Completeness': {'score': 2.0, 'feedback': 'yyy'}},
  'session_feedback': 'zzz'},
 {'section': 'Education',
  'total_section_raw_score': 20.0,
  'total_section_max_score': 20.0,
  'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
   'Completeness': {'score': 10.0, 'feedback': 'def'}},
  'session_feedback': 'ghi'}]

In [90]:
# def fn1():
def aggregate_weighted_section_scores():
    weights      = weight_config["weights"]
    contribution = {}                    # Keep stat of score and detail
    total_weighted_raw_score = 0.0       # Accumulate raw score after time by weight
    total_weighted_max_score = 0.0       # Accumulate max score after time by weight
    for section_data in section_outputs: # Loop with section_output (Ss)
        section_name            = section_data["section"]   # Get section name
        total_section_raw_score = section_data["total_section_raw_score"]    # Get raw section score from each section
        total_section_max_score = section_data["total_section_max_score"]    # Get max section score from each section
        section_weight          = weights[section_name]["section_weight"]    # Get section_weight from weight.yaml e.g. 0.1,0.2
        
        total_section_raw_score_x_weight = total_section_raw_score*section_weight   # raw_score x weight
        total_section_max_score_x_weight = total_section_max_score*section_weight   # max_score x weight

        contribution[section_name] = {
            "total_section_raw_score":total_section_raw_score,
            "total_section_max_score":total_section_max_score,
            "section_weight":section_weight,
            "total_section_raw_score_x_weight":total_section_raw_score_x_weight,
            "total_section_max_score_x_weight":total_section_max_score_x_weight
        }
        total_weighted_raw_score  = total_weighted_raw_score + total_section_raw_score_x_weight   # E(raw_score x weight)
        total_weighted_max_score  = total_weighted_max_score + total_section_max_score_x_weight   # E(max_score x weight)
        
    return {
        "total_weighted_raw_score":total_weighted_raw_score, # E(raw_score x weight) Summation of raw score for every section every criteria
        "total_weighted_max_score":total_weighted_max_score, # E(max_score x weight) Summation of max score for every section every criteria
        "section_contribution":contribution                  # Details
    }


In [91]:
test01 = aggregate_weighted_section_scores()
test01

{'total_weighted_raw_score': 18.4,
 'total_weighted_max_score': 20.0,
 'section_contribution': {'Profile': {'total_section_raw_score': 4.0,
   'total_section_max_score': 20.0,
   'section_weight': 0.1,
   'total_section_raw_score_x_weight': 0.4,
   'total_section_max_score_x_weight': 2.0},
  'Education': {'total_section_raw_score': 20.0,
   'total_section_max_score': 20.0,
   'section_weight': 0.9,
   'total_section_raw_score_x_weight': 18.0,
   'total_section_max_score_x_weight': 18.0}}}

<hr>
<hr>

In [92]:
def normalize_score(total_raw_score,total_max_score):
    print(f"total_raw_score -> {total_raw_score}")
    print(f"total_max_score -> {total_max_score}")
    normalize_score = (total_raw_score/total_max_score) * 100
    print(f"normalize_score -> {normalize_score}/100")
    return normalize_score

def score_to_grade(score: float) -> str:
    if 90 <= score <= 100:
        return "A"
    elif 80 <= score < 90:
        return "B"
    elif 70 <= score < 80:    
        return "C"
    elif 60 <= score < 70:
        return "D"
    elif 0 <= score < 60:
        return "F"
    else:
        return "Gradding error"

In [93]:
norm_score = normalize_score(test01["total_weighted_raw_score"],test01["total_weighted_max_score"])
score_to_grade(norm_score)

total_raw_score -> 18.4
total_max_score -> 20.0
normalize_score -> 92.0/100


'A'